# Where is it, exactly?

**Lecture 17 · Build** · Géron, Chapter 12

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** Read before you run. The cell marked
**⚠ read before running** contains a defect on purpose, and it is the defect
this lecture is about: it runs, it prints a believable number, and the number
is wrong by a factor of nine.

**The corpus is 128 images.** COCO's `val2017` split is 5,000 images and the
full release is about 20 GB. Neither is downloaded here. Every number this
notebook prints is a measurement on 128 images, and you are expected to say
"128 images" whenever you quote one.

**About the prompt boxes.** Where a code cell is preceded by a quoted prompt,
three lines follow it: what the prompt leaves open, the version a student
typically writes instead, and how you would catch a wrong answer. Those three
lines are the part worth reading twice.

The prompts here are **specifications, not transcripts** — this is what you
would have to ask for in order to get this cell, not a recording of somebody
asking for it. If your own prompt is vaguer than the box, expect worse code than
the cell below it.

*Lecture 19 is the one exception in this course.* It was built cell by cell
against Colab's Gemini 3.1 Pro, and its prompts are verbatim. It says so itself.


## 1 · Setup

> **Prompt · setup**
>
> **input** · nothing
>
> **output** · versions, one seed, the device, and the corpus size as a named constant
>
> **constraint** · `N_IMAGES = 128` as a NAMED constant with a comment saying to say it out loud — every number this notebook prints is a measurement on 128 images

**Watch this prompt.**

* **Left open:** that COCO's val2017 split is 5,000 images and the full release is about 20 GB. Neither is downloaded here, and the notebook says so in three places.
* **The usual student version:** quoting a number from this notebook as 'the COCO result'. It is a 128-image result, and the difference is a factor of forty in sample size.
* **How you would catch it:** when a notebook works on a subset, put the subset size in a constant with a name, not a literal in a slice. It then appears in every printout that uses it.

In [ ]:
# --- setup -------------------------------------------------------------------
# Not examinable: this is engineering hygiene, not machine learning.
import sys, json, time, itertools, urllib.request, zipfile, io
from pathlib import Path

import numpy as np
import torch, torchvision
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

print(f"python       {sys.version.split()[0]}")
print(f"torch        {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")

RANDOM_STATE = 42                  # one seed, used everywhere
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"device       {DEVICE}")

N_IMAGES = 128                     # the corpus. Say it out loud every time.
DATA = Path("datasets/coco")
DATA.mkdir(parents=True, exist_ok=True)

## 2 · The corpus, and exactly how big it is

> **Prompt · ⏱ 60-90 s — the corpus, and exactly how big it is**
>
> **input** · COCO's annotation file and 128 JPEGs
>
> **output** · the annotation blob, the 128 chosen images, and the files on disk
>
> **constraint** · choose the images by a RULE — the 128 numerically lowest image ids — so that nobody chose which images make the detector look good
>
> **check** · assert exactly N_IMAGES were selected

**Watch this prompt.**

* **Left open:** that the annotation file is 241 MB and is the larger of the two downloads. It is the only way to have real ground-truth boxes at all.
* **The usual student version:** picking images that look interesting, or the first 128 in file order, which on COCO is not the same as id order and is not reproducible across mirrors.
* **How you would catch it:** a selection rule you can state in one sentence is a selection rule somebody can check. 'The lowest 128 ids' is; 'a representative sample' is not.

In [ ]:
# --- the data ----------------------------------------------------------------
# Two downloads. The annotation file is the larger of them and it is the only
# way to have real ground-truth boxes at all; the images are 128 JPEGs, not
# 5,000 and certainly not the 20 GB training split.
#
# ⏱ about 60-90 seconds the first time, instant afterwards.
ANN = DATA / "instances_val2017.json"
IMG_DIR = DATA / "images"
IMG_DIR.mkdir(exist_ok=True)

if not ANN.is_file():
    url = ("http://images.cocodataset.org/annotations/"
           "annotations_trainval2017.zip")
    print(f"downloading annotations (~241 MB) from {url}")
    blob = urllib.request.urlopen(url).read()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        ANN.write_bytes(z.read("annotations/instances_val2017.json"))

raw = json.loads(ANN.read_text())
print(f"{len(raw['images']):,} images in val2017, "
      f"{len(raw['categories'])} categories")

# The 128 numerically lowest image ids. A rule, not a selection: nobody chose
# which images make the detector look good.
images = sorted(raw["images"], key=lambda i: i["id"])[:N_IMAGES]
ids = {im["id"] for im in images}
assert len(images) == N_IMAGES

for im in images:
    p = IMG_DIR / im["file_name"]
    if not p.is_file():
        urllib.request.urlretrieve(
            "http://images.cocodataset.org/val2017/" + im["file_name"], p)
print(f"{len(list(IMG_DIR.glob('*.jpg')))} images on disk")

### 2.1 · Ground truth

Two things to notice in the next cell, both of which cost people an afternoon
the first time:

1. COCO stores a box as `[x, y, w, h]`; torchvision returns `[x1, y1, x2, y2]`.
   Convert once, at the edge of the program.
2. `iscrowd = 1` means the annotator drew one region around many instances
   rather than boxing them separately. Dropping those is a *choice*, it changes
   every count below, and this is where it is recorded.

> **Prompt · ground truth, converted once at the edge**
>
> **input** · COCO's annotations for the 128 images
>
> **output** · corner-form boxes and labels per image, with the crowd regions counted and dropped
>
> **constraint** · COCO stores [x, y, w, h] and torchvision emits [x1, y1, x2, y2]. Convert HERE and nowhere else — from this cell on, every box in memory is corners
>
> **check** · assert x2 ≥ x1 and y2 ≥ y1 on every box, which is exactly the assertion that fires if w was read as x2

**Watch this prompt.**

* **Left open:** that dropping `iscrowd=1` is a CHOICE. One polygon drawn around many instances is not one object and not n objects — it is a refusal to decide — and it changes every count below.
* **The usual student version:** mixing the two conventions, which is the commonest bug in this material. A [x,y,w,h] box read as corners has x2 = width, so it is a box in the top-left corner and the IoU against it is near zero for everything.
* **How you would catch it:** convert at the boundary of your program, assert the invariant immediately, and never carry two conventions in the same variable name.

In [ ]:
# --- ground truth, converted once, at the edge --------------------------------
# COCO stores [x, y, w, h]. torchvision emits [x1, y1, x2, y2]. Mixing them is
# the commonest bug in this material, so the conversion happens HERE and
# nowhere else; from this cell on, every box in memory is corners.
cat_name = {c["id"]: c["name"] for c in raw["categories"]}

gt = {i: {"boxes": [], "labels": []} for i in ids}
n_crowd = 0
for a in raw["annotations"]:
    if a["image_id"] not in ids:
        continue
    if a["iscrowd"]:
        # One polygon drawn around many instances. Not one object, not n
        # objects — a refusal to decide. COCO's own evaluator ignores them.
        n_crowd += 1
        continue
    x, y, w, h = a["bbox"]
    gt[a["image_id"]]["boxes"].append([x, y, x + w, y + h])
    gt[a["image_id"]]["labels"].append(a["category_id"])

for iid, g in gt.items():
    g["boxes"] = np.asarray(g["boxes"], dtype=float).reshape(-1, 4)
    g["labels"] = np.asarray(g["labels"], dtype=np.int64)

# assert, do not hope
for iid, g in gt.items():
    assert (g["boxes"][:, 2] >= g["boxes"][:, 0]).all(), "x2 < x1: w read as x2"
    assert (g["boxes"][:, 3] >= g["boxes"][:, 1]).all(), "y2 < y1: same bug"

n_true = np.array([len(gt[im["id"]]["labels"]) for im in images])
assert n_true.shape == (N_IMAGES,)
print(f"{N_IMAGES} images, {n_true.sum()} objects, "
      f"{n_crowd} crowd regions dropped")
print(f"objects per image: mean {n_true.mean():.2f}  median "
      f"{np.median(n_true):.0f}  range {n_true.min()}-{n_true.max()}")

### 2.2 · What is in it

`person` dominates. Remember that: in the next lecture we start averaging over
categories, and a mean over categories does not care that one of them is 39% of
the corpus.

> **Prompt · what is in it**
>
> **input** · the ground-truth labels
>
> **output** · how many categories appear, and the eight commonest
>
> **constraint** · print `person` as a SHARE of every annotated object, not just as a count

**Watch this prompt.**

* **Left open:** why that share matters later. The next lecture starts averaging over categories, and a mean over categories does not care that one of them is 39% of the corpus.
* **The usual student version:** not looking at the class distribution, then being surprised when a per-category mean and an overall figure disagree wildly.
* **How you would catch it:** only some of the 80 categories appear in 128 images. Any per-category metric will have empty categories in it, and what you do about those changes the mean.

In [ ]:
import collections

freq = collections.Counter()
for g in gt.values():
    for c in g["labels"]:
        freq[cat_name[int(c)]] += 1

print(f"{len(freq)} of 80 categories appear in these {N_IMAGES} images\n")
for name, k in freq.most_common(8):
    print(f"  {name:14s} {k:4d}")
print(f"\nperson is {freq['person'] / n_true.sum():.1%} of every "
      f"annotated object")

## 3 · A metric, and the baseline that kills the obvious one

The obvious metric is: *a detection is correct when its box overlaps the true
box.* It is computable, unambiguous and parameter-free.

Before adopting any metric, this course computes what the stupidest possible
system scores under it. For detection, the stupidest possible system is
**one box per image, covering the whole image**.

> **Prompt · the baseline that kills the obvious metric**
>
> **input** · one box per image, covering the whole image
>
> **output** · how many true objects it overlaps
>
> **constraint** · test the OBVIOUS metric — a detection is correct when its box overlaps the true box — against the stupidest possible system before adopting it
>
> **check** · assert it hits every object, which also verifies no annotated box lies outside its own image

**Watch this prompt.**

* **Left open:** that the metric is dead once this prints 100%. It rewards a box for being enormous and nothing in it punishes size.
* **The usual student version:** adopting overlap-based matching because it is computable, unambiguous and parameter-free. All three are true and it is still worthless.
* **How you would catch it:** compute what the stupidest possible system scores under any metric BEFORE adopting it. Here that takes six lines and rules out the obvious choice.

In [ ]:
def overlaps(a, b):
    """Do two corner-form boxes share any area at all?"""
    lt = np.maximum(a[:2], b[:2])
    rb = np.minimum(a[2:], b[2:])
    wh = np.clip(rb - lt, 0.0, None)
    return bool(wh[0] * wh[1] > 0)

hits = total = 0
for im in images:
    whole = np.array([0.0, 0.0, float(im["width"]), float(im["height"])])
    for b in gt[im["id"]]["boxes"]:
        hits += overlaps(whole, b)
        total += 1

print(f"the whole-image box overlaps {hits} of {total} true objects "
      f"= {hits / total:.1%}")
assert hits == total, "if this ever fails, a box lies outside its own image"

**100%.** A system with no weights, no data and no idea scores perfectly under
the proposed metric. That metric is dead: it rewards a box for being enormous,
and nothing in it punishes size.

So today's metric is the one thing left that the whole-image box loses at:
**counting**.

> **Prompt · the metric we can defend**
>
> **input** · predicted and true object counts
>
> **output** · the mean absolute error, and two trivial baselines
>
> **constraint** · assert the two arrays have the same shape — a count vector of the wrong length broadcasts silently and gives a plausible number

**Watch this prompt.**

* **Left open:** that counting is the one thing left that the whole-image box loses at. It is not a good metric; it is the one that survives the baseline test.
* **The usual student version:** comparing per-image counts against a scalar mean and getting a number that means nothing, because numpy broadcast it.
* **How you would catch it:** a system that never opens the image scores 6.02. Every number below has to be read against that.

In [ ]:
def count_mae(pred_counts, true_counts):
    pred_counts = np.asarray(pred_counts)
    assert pred_counts.shape == true_counts.shape
    return float(np.abs(pred_counts - true_counts).mean())

one_box   = count_mae(np.ones(N_IMAGES), n_true)
mean_box  = count_mae(np.full(N_IMAGES, round(n_true.mean())), n_true)

print(f"one box per image           MAE {one_box:.2f}")
print(f"predict the corpus mean     MAE {mean_box:.2f}")
print(f"perfect                     MAE 0.00")

## 4 · Commit

**Stop. On paper, now.** Not in this notebook — on paper, where you cannot
quietly revise it.

```
Metric:                                              ____________
Count MAE a useful shelf-audit system would need:    ____________
Count MAE I expect from what we build today:         ____________
```

You are not guessing in the dark: a system that never opens the image scores
6.02, and perfect is 0.00. Saying *where between them* is the exercise.

## 5 · The detector

Nothing is trained here. These are the weights torchvision ships, trained on
COCO's training split by someone else, and this lecture is about evaluating
them rather than fitting them.

⏱ the weights are about 167 MB; the download happens once.

> **Prompt · the detector — nothing is trained here**
>
> **input** · torchvision's COCO-trained weights
>
> **output** · the model in eval mode, its preprocessing, and its category names
>
> **constraint** · assert that the model's label integers ARE COCO's category_ids — the comparison further down is only legitimate if they agree
>
> **check** · the name-agreement assert, plus `names[1] == 'person'`

**Watch this prompt.**

* **Left open:** that there are 91 label slots for 80 categories. COCO's ids are not contiguous, and the gaps are why the assert is worth writing rather than assuming.
* **The usual student version:** building a mapping from the model's index order to COCO ids by hand. It is unnecessary here and it is wrong in a way that shifts every category by a few positions.
* **How you would catch it:** print the weights' own reported metrics on the full 5,000 images. Your 128-image number should be read next to it, not instead of it.

In [ ]:
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights)

weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval().to(DEVICE)              # eval(), every time — Lecture 12
preprocess = weights.transforms()

names = weights.meta["categories"]
print(f"{len(names)} label slots for 80 categories")
print("slot 0 is", names[0], "| slot 12 is", names[12])
assert names[1] == "person"
# the integer in `labels` is the same integer as COCO's category_id, which is
# the only reason the comparison further down is legitimate
assert all(names[cid] == nm for cid, nm in cat_name.items())

print("\ntorchvision's own reported score for these weights,")
print("on all 5,000 val2017 images:", weights.meta["_metrics"])

### 5.1 · Run it

⏱ **1 to 2 minutes** on a GPU or an Apple Silicon MPS backend, and several
minutes on a CPU-only runtime. It varies with what else the machine is doing:
the same loop took 39 s on an idle laptop and 102 s on a busy one. No output
does not mean it has hung.

> **Prompt · ⏱ 1-2 min on GPU, longer on CPU — run it**
>
> **input** · the 128 images
>
> **output** · predictions per image, and the wall clock per image
>
> **constraint** · `torch.inference_mode()` — no graph and no gradients, which matters here because a detector's intermediate tensors are large
>
> **check** · assert one prediction per image

**Watch this prompt.**

* **Left open:** that the timing varies with what else the machine is doing. The same loop took 39 s on an idle laptop and 102 s on a busy one, and no output does not mean it has hung.
* **The usual student version:** using `no_grad()` and wondering about the memory. `inference_mode` is stronger — it also skips version counting — and for a pure evaluation loop there is no reason not to.
* **How you would catch it:** move the predictions to CPU numpy inside the loop. Keeping 128 sets of GPU tensors alive is how the next cell fails for a reason that has nothing to do with the next cell.

In [ ]:
t0 = time.time()
preds = {}
with torch.inference_mode():                 # no graph, no gradients
    for im in images:
        img = Image.open(IMG_DIR / im["file_name"]).convert("RGB")
        out = model([preprocess(img).to(DEVICE)])[0]
        preds[im["id"]] = {k: v.cpu().numpy() for k, v in out.items()}
elapsed = time.time() - t0

assert len(preds) == N_IMAGES
print(f"{N_IMAGES} images in {elapsed:.1f} s "
      f"({elapsed / N_IMAGES:.2f} s per image on {DEVICE})")

### 5.2 · Read the shape before you read the answer

> **Prompt · read the shape before the answer**
>
> **input** · one image's predictions
>
> **output** · the shape and dtype of every returned array
>
> **constraint** · assert the three arrays are the same length AND that scores come back sorted descending — everything below relies on both

**Watch this prompt.**

* **Left open:** the number itself: this image has far more boxes than it has annotated objects. That is reviewer question 3 answering reviewer question 5 before it is asked.
* **The usual student version:** going straight to the counting. A shape of 88 for a photograph with twenty annotated objects should stop you, and it is visible one cell before the bug.
* **How you would catch it:** assert the sort order rather than assuming it. Several detection APIs return unsorted boxes, and code that slices the 'top k' silently takes an arbitrary k.

In [ ]:
p = preds[images[0]["id"]]
for k, v in p.items():
    print(f"{k:8s} {v.shape} {v.dtype}")

assert p["boxes"].shape[0] == p["labels"].shape[0] == p["scores"].shape[0]
assert np.all(np.diff(p["scores"]) <= 0), "not sorted by score"
print(f"\nthis image has {len(p['boxes'])} boxes and "
      f"{len(gt[images[0]['id']]['labels'])} annotated objects")

## 6 · An assistant writes the counting code

> *"Use a pretrained Faster R-CNN from torchvision to count how many objects
> are in each image in this folder, and print the mean absolute error against
> the COCO annotations."*

**⚠ Read before running.** Everything in that request is true and none of it is
wrong. One constraint is missing. Find it before you scroll.

> **Prompt · ⚠ what the assistant returns**
>
> **input** · 'count how many objects are in each image and print the MAE against the COCO annotations'
>
> **output** · the mean count and the MAE
>
> **constraint** · run it exactly as written — everything in that request is true and none of it is wrong. ONE constraint is missing

**Watch this prompt.**

* **Left open:** reviewer question 5. `len(pred['boxes'])` is the number of rows the model CHOSE to return: everything above `box_score_thresh`, which defaults to 0.05, capped at `box_detections_per_img`, which defaults to 100. It is not a count of objects, it is a count of CANDIDATES.
* **The usual student version:** this exact line. It runs, it prints a plausible number, and the number is wrong by a factor of nine.
* **How you would catch it:** whenever a library hands you a variable-length result, ask what decided the length. Two defaults decided this one and neither was mentioned in the prompt or the output.

In [ ]:
# the code the request returns — it runs, and it prints a plausible number
counts_naive = np.array([len(preds[im["id"]]["boxes"]) for im in images])

print(f"mean objects per image: {counts_naive.mean():.2f}")
print(f"count MAE: {count_mae(counts_naive, n_true):.2f}")

### Reviewer question 5: what is the default I did not ask for?

`len(pred["boxes"])` is the number of rows the model chose to return. That is
everything above `box_score_thresh`, which defaults to **0.05**, capped at
`box_detections_per_img`, which defaults to **100**. It is not a count of
objects. It is a count of *candidates*.

Reviewer question 3 — *what is the shape here?* — finds it too: a shape of 88
for a photograph with twenty annotated objects in it should stop you.

**Now measure the damage.** Do not estimate it.

> **Prompt · measure the damage, do not estimate it**
>
> **input** · every score the model returned
>
> **output** · how many are below 0.10, how many at or above 0.50, the median, and whether the per-image cap was hit
>
> **constraint** · show that the CAP was reached — 100 boxes for one image is the default `box_detections_per_img`, not a property of the photograph

**Watch this prompt.**

* **Left open:** that the median score is low enough to make the point on its own. Half the returned boxes are things the model barely believes in.
* **The usual student version:** guessing that the threshold 'probably doesn't matter much'. The distribution is in memory and takes four lines to summarise.
* **How you would catch it:** a count that hits a round number like exactly 100 is almost always a cap rather than a measurement. Look it up before interpreting it.

In [ ]:
scores = np.concatenate([preds[im["id"]]["scores"] for im in images])
print(f"boxes returned in total: {len(scores):,}")
print(f"  below score 0.10: {(scores < 0.10).sum():,}")
print(f"  at or above 0.50: {(scores >= 0.50).sum():,}")
print(f"  median score:     {np.median(scores):.3f}")

# the model's own cap, hit
print(f"\nmost boxes returned for one image: {counts_naive.max()} "
      f"(box_detections_per_img is 100)")

### The corrected specification

> *"… count how many objects are in each image, **keeping only detections whose
> score is at least a threshold I pass in**. Print the mean absolute error
> against the annotations, **and print the trivial baseline of one box per image
> beside it**. **Assert that the number kept never exceeds the number
> returned.**"*

Three additions: name the parameter, demand the baseline, assert the
relationship. The third is the one that fails loudly.

> **Prompt · the corrected version**
>
> **input** · the predictions and an explicit threshold
>
> **output** · the count MAE, the signed bias, and the baseline
>
> **constraint** · NO DEFAULT on the threshold — a count of objects is meaningless without saying which detections were counted
>
> **check** · assert the kept count never exceeds the returned count, and assert the MAE beats the one-box-per-image baseline

**Watch this prompt.**

* **Left open:** that THRESH = 0.5 was chosen BEFORE looking at the error curve. The comment says so, and the next section is about what happens if it is not.
* **The usual student version:** giving the function a default of 0.5 to make it convenient. The convenience is exactly what let the bug through one cell earlier.
* **How you would catch it:** the signed bias beside the absolute one. A MAE of 3.00 that is entirely over-counting is a different system from one that is balanced, and only the signed figure distinguishes them.

In [ ]:
def count_objects(pred, thresh):
    """Objects detected at or above `thresh`.

    There is deliberately no default: a count of objects is meaningless
    without saying which detections were counted.
    """
    keep = pred["scores"] >= thresh
    assert keep.sum() <= len(pred["scores"])
    return int(keep.sum())

THRESH = 0.5                     # chosen BEFORE looking at the error curve
n_pred = np.array([count_objects(preds[im["id"]], THRESH) for im in images])

mae = count_mae(n_pred, n_true)
bias = float((n_pred - n_true).mean())
print(f"count MAE  {mae:.2f}")
print(f"signed     {bias:+.2f}   (positive = too many boxes)")
print(f"baseline   {one_box:.2f}")
assert mae < one_box, "worse than predicting one box per image"

The naive version was **27.51**, this one is **3.00**, and the baseline that
never opens the image is **6.02**. One missing line took the answer from "half
the error of a system that ignores the picture" to "four times worse than one".

Nothing raised. Nothing warned. The number just looked plausible.

## 7 · The threshold is a knob, and nobody chose it

Sweep it and watch the answer to the stakeholder's question move by a factor of
ten.

> **Prompt · the threshold is a knob, and nobody chose it**
>
> **input** · nineteen thresholds
>
> **output** · mean count and MAE at each, with the reported one marked
>
> **constraint** · mark the value we report, and find the BEST one — then refuse to report the best

**Watch this prompt.**

* **Left open:** why we refuse. The best threshold was found on the same 128 images we then report on, which is choosing a hyperparameter on the test set — the failure from application 3, wearing a detection costume.
* **The usual student version:** reporting the minimum of the sweep. It is lower, it is honestly computed, and it is selection on the evaluation set.
* **How you would catch it:** the answer to the stakeholder's question moves by a factor of ten across this sweep. A single count with no threshold stated is not an answer.

In [ ]:
ts = np.round(np.arange(0.05, 0.96, 0.05), 2)
rows = []
for t in ts:
    c = np.array([count_objects(preds[im["id"]], t) for im in images])
    rows.append((t, c.mean(), count_mae(c, n_true)))

print(f"{'thresh':>7s} {'mean/img':>9s} {'MAE':>7s}")
for t, m, e in rows:
    mark = "  <- we report this" if abs(t - THRESH) < 1e-9 else ""
    print(f"{t:7.2f} {m:9.2f} {e:7.2f}{mark}")

best = min(rows, key=lambda r: r[2])
print(f"\nlowest MAE is {best[2]:.2f} at threshold {best[0]:.2f}")
print("We do NOT report that one: it was found on the same 128 images we")
print("then report on, which is choosing a hyperparameter on the test set.")

## 8 · Look at the pictures, not only at the number

Three images with their predicted boxes. Labels are drawn only for confident
detections, because a crowded image stacks fourteen captions on top of each
other and an illegible figure teaches nothing.

> **Prompt · look at the pictures, not only at the number**
>
> **input** · three images with their detections
>
> **output** · boxes drawn, with labels only on confident detections
>
> **constraint** · label only above a high score — a crowded image stacks fourteen captions on top of each other and an illegible figure teaches nothing

**Watch this prompt.**

* **Left open:** what you will see: two boxes on one object, one a little too large, one confident about nothing at all. Some of them are visibly wrong.
* **The usual student version:** drawing every label, producing a figure that is unreadable, and concluding the visual check is not worth doing.
* **How you would catch it:** try to write down a number for 'how wrong is that box'. You cannot, and neither can the metric you committed to.

In [ ]:
def show(iid, thresh=THRESH, label_above=0.90, ax=None):
    im = next(i for i in images if i["id"] == iid)
    ax = ax or plt.gca()
    ax.imshow(Image.open(IMG_DIR / im["file_name"]).convert("RGB"))
    p = preds[iid]
    keep = np.flatnonzero(p["scores"] >= thresh)
    for k in keep:
        x1, y1, x2, y2 = p["boxes"][k]
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                               edgecolor="#c0392b", linewidth=2))
        if p["scores"][k] >= label_above:
            ax.text(x1 + 2, max(y1 - 4, 12),
                    f"{names[int(p['labels'][k])]} {p['scores'][k]:.2f}",
                    color="white", fontsize=8,
                    bbox=dict(fc="#c0392b", ec="none", pad=1.0))
    ax.set_title(f"{len(keep)} boxes, {len(gt[iid]['labels'])} true",
                 fontsize=10, loc="left")
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, im in zip(axes, images[:3]):
    show(im["id"], ax=ax)
plt.tight_layout(); plt.show()

Some of those boxes are visibly wrong: two on one object, one a little too
large, one confident about nothing at all.

**You have no way to say how wrong.** Try it: write down a number for "how
wrong is that box". You cannot, and neither can the metric you committed to.

> **Prompt · two systems the metric cannot tell apart**
>
> **input** · nine true boxes, and the same nine shifted 400 pixels
>
> **output** · the count error of each
>
> **constraint** · construct the counterexample rather than describing it — both systems emit nine boxes for an image with nine objects, and one puts them on the objects

**Watch this prompt.**

* **Left open:** that this is a proof, not an illustration. The committed metric scores both perfect, so no amount of tuning it can distinguish them.
* **The usual student version:** accepting 'counting is a weak metric' as a general remark. Two arrays and three prints turn it into something you cannot argue with.
* **How you would catch it:** when you suspect a metric is blind to something, build the pair it cannot separate. If you can, the metric is dead for that purpose.

In [ ]:
# Two systems your metric cannot tell apart. Both emit nine boxes for an
# image with nine objects; one puts them on the objects and one does not.
true_boxes = gt[images[0]["id"]]["boxes"]
k = min(9, len(true_boxes))

system_a = true_boxes[:k].copy()                  # exactly right
system_b = true_boxes[:k].copy() + 400.0          # exactly the wrong places

print(f"system A: {len(system_a)} boxes, count error "
      f"{abs(len(system_a) - k)}")
print(f"system B: {len(system_b)} boxes, count error "
      f"{abs(len(system_b) - k)}")
print("\nYour committed metric scores both of them perfect.")

## 9 · Propose the missing number

Whatever repairs this has to:

1. be 1 for identical boxes and 0 for boxes that do not touch
2. punish a box for being **too large**, or the whole-image baseline wins again
3. punish a box for being **too small**, or a one-pixel box in the right place
   wins
4. be dimensionless, so a 40-pixel cup and a 400-pixel sofa are on one scale

Write yours as a function of two corner-form boxes, and test it on two
identical boxes and on two that do not touch.

> **Prompt · propose the missing number**
>
> **input** · two corner-form boxes
>
> **output** · your own overlap score, tested on an identical pair and a disjoint pair
>
> **constraint** · four requirements: 1 for identical boxes, 0 for disjoint ones, punishes too-large AND too-small, and dimensionless so a 40-pixel cup and a 400-pixel sofa are on one scale

**Watch this prompt.**

* **Left open:** deliberately everything. The body raises NotImplementedError and it is yours to write before the next lecture.
* **The usual student version:** satisfying three of the four. Dropping requirement 2 lets the whole-image box win again; dropping 3 lets a one-pixel box in the right place win.
* **How you would catch it:** if your formula does something odd for the DISJOINT pair, do not fix it. That is the interesting case, and the next lecture is largely about it.

In [ ]:
def my_box_score(a, b):
    """Your formula. Replace the body.

    Requirements: 1 when a == b, 0 when disjoint, punishes both too-large and
    too-small, dimensionless.
    """
    raise NotImplementedError("this is yours to write")

same = np.array([0.0, 0.0, 100.0, 100.0])
away = np.array([300.0, 0.0, 400.0, 100.0])

try:
    print("identical:", my_box_score(same, same))
    print("disjoint: ", my_box_score(same, away))
except NotImplementedError as exc:
    print("not written yet —", exc)
    print("\nBring your version to the next lecture. If it does something odd")
    print("for the disjoint pair, do NOT fix it. That is the interesting case.")

## 10 · Where we are

| System | Count MAE, 128 images |
|---|---|
| One box per image | 6.02 |
| Every box the model returns | 27.51 |
| Faster R-CNN at score ≥ 0.5 | **3.00** |

Write **3.00** next to what you predicted, and keep the sheet.

Do not fix anything. Counting cannot distinguish nine right boxes from nine
wrong ones, and the repair is the next ninety minutes.